# DATA101 Research Project:
# Trends in Literacy Proficiency Within the U.S. in Recent Years (2017-2023)
## Maximilian Shin


### Topic: 



Data Sources:
- (PIAAC) Program for the International Assessment of Adult Competencies
    - oecd.org 
    - National Center for Education Statistics
    - American Time Use Survey

Other Research Sources:
- National Assessment of Educational Progress
    - (changes in literacy proficiency in younger demographics not covered by other surveys, i.e. <15 years of age)
- World Bank Group Data
    - (global literacy rates)


In [22]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as st
from statsmodels.stats.proportion import proportions_ztest, proportion_confint, proportion_effectsize
from statsmodels.stats.power import NormalIndPower, TTestIndPower, TTestPower

### Importing Data

cycle1_PIAAC1 = pd.read_sas("prgusap1_2012.sas7bdat")
print(cycle1_PIAAC1.shape)
print(cycle1_PIAAC1.info())
print()

cycle1_PIAAC2 = pd.read_sas("prgusap1_2017.sas7bdat")
print(cycle1_PIAAC2.shape)
print(cycle1_PIAAC2.info())
print()

cycle2_PIAAC = pd.read_sas("prgusap2.sas7bdat")
print(cycle2_PIAAC.shape)
print(cycle2_PIAAC.info())
print()


(5010, 1328)
<class 'pandas.DataFrame'>
RangeIndex: 5010 entries, 0 to 5009
Columns: 1328 entries, CNTRYID to WRITWORK_WLE_CA
dtypes: float64(1309), object(19)
memory usage: 50.8+ MB
None

(3660, 1349)
<class 'pandas.DataFrame'>
RangeIndex: 3660 entries, 0 to 3659
Columns: 1349 entries, CNTRYID to EARNMTHSELFPPPUS_C
dtypes: float64(1327), object(22)
memory usage: 37.7+ MB
None

(3765, 2483)
<class 'pandas.DataFrame'>
RangeIndex: 3765 entries, 0 to 3764
Columns: 2483 entries, SEQID to SPFWT80
dtypes: float64(2444), object(39)
memory usage: 71.3+ MB
None



(3660, 1349)
<class 'pandas.DataFrame'>
RangeIndex: 3660 entries, 0 to 3659
Columns: 1349 entries, CNTRYID to EARNMTHSELFPPPUS_C
dtypes: float64(1327), object(22)
memory usage: 37.7+ MB
None


,CNTRYID,CNTRY,CNTRYID_E,CNTRY_E,SEQID,GENDER_R,DISP_CIBQ,A_N01_T,B_Q01A,B_Q01A_T,...,EARNHRPPPUS_C,EARNHRBONUSUS_C,EARNHRBONUSPPPUS_C,EARNMTHUS_C,EARNMTHPPPUS_C,EARNMTHBONUSUS_C,EARNMTHBONUSPPPUS_C,EARNMTHALLUS_C,EARNMTHALLPPPUS_C,EARNMTHSELFPPPUS_C
0,84093.0,b'USA',840.0,b'USA',9355.0,1.0,1.0,1.0,7.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,84093.0,b'USA',840.0,b'USA',10882.0,1.0,1.0,1.0,13.0,3.0,...,100.0,100.0,100.0,23833.333333,23833.333333,23833.333333,23833.333333,23833.333333,23833.333333,NaN
2,84093.0,b'USA',840.0,b'USA',12329.0,2.0,1.0,2.0,7.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,84093.0,b'USA',840.0,b'USA',10711.0,2.0,1.0,2.0,7.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1666.666667,1666.666667,1666.666667
4,84093.0,b'USA',840.0,b'USA',10383.0,1.0,1.0,1.0,7.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4166.666667,4166.666667,4166.666667


In [15]:
sample2012 = cycle1_PIAAC1.loc[:, ["GENDER_R", "AGEG10LFS_T", "AGEG10LFS", "HOMLANG", "PBROUTE", "PRC_PV_SCR", "PRC_SP_SCR", "PRC_PC_SCR", 
                                   "LITSTATUS", "PVLIT1", "PVLIT2", "PVLIT3", "PVLIT4", "PVLIT5", "PVLIT6", "PVLIT7", "PVLIT8", "PVLIT9", "PVLIT10",
                                   "READHOME", "READHOME_WLE_CA", "READWORK", "READWORK_WLE_CA", 
                                   "WRITHOME", "WRITHOME_WLE_CA", "WRITWORK", "WRITWORK_WLE_CA",]]
sample2012.head(20)


,GENDER_R,AGEG10LFS_T,AGEG10LFS,HOMLANG,PBROUTE,PRC_PV_SCR,PRC_SP_SCR,PRC_PC_SCR,LITSTATUS,PVLIT1,...,PVLIT9,PVLIT10,READHOME,READHOME_WLE_CA,READWORK,READWORK_WLE_CA,WRITHOME,WRITHOME_WLE_CA,WRITWORK,WRITWORK_WLE_CA
0,1.0,3.0,3.0,1.0,4.0,NaN,NaN,NaN,1.0,256.53546,...,251.34951,251.24423,1.414224,2.0,2.141952,3.0,1.726160,2.0,1.288079,2.0
1,1.0,2.0,2.0,0.0,4.0,NaN,NaN,NaN,1.0,201.21161,...,176.47494,196.24402,1.753158,2.0,1.130051,1.0,1.070335,2.0,2.022641,3.0
2,2.0,5.0,5.0,1.0,4.0,NaN,NaN,NaN,1.0,281.99606,...,299.25286,278.00204,2.885365,5.0,NaN,NaN,2.670381,4.0,NaN,NaN
3,1.0,3.0,3.0,1.0,4.0,NaN,NaN,NaN,1.0,327.51296,...,322.73357,291.86959,2.164301,3.0,2.718257,4.0,2.446716,3.0,1.946433,3.0
4,2.0,5.0,5.0,1.0,2.0,34.0,18.0,41.0,1.0,239.66491,...,201.97481,231.42871,3.195616,5.0,NaN,NaN,2.574343,4.0,NaN,NaN
5,1.0,3.0,3.0,1.0,4.0,NaN,NaN,NaN,1.0,246.67828,...,266.70124,224.74148,4.265535,5.0,3.278885,5.0,3.427300,5.0,0.550923,1.0
6,1.0,4.0,4.0,1.0,4.0,NaN,NaN,NaN,1.0,279.72735,...,264.76856,292.53873,3.697555,5.0,1.627924,2.0,2.222770,3.0,NaN,0.0
7,1.0,4.0,4.0,1.0,4.0,NaN,NaN,NaN,1.0,355.77880,...,377.26111,385.74618,1.928241,3.0,2.241920,3.0,2.642440,4.0,2.293246,4.0
8,2.0,4.0,4.0,1.0,4.0,NaN,NaN,NaN,1.0,248.25320,...,245.45555,269.66448,3.175204,5.0,2.489037,4.0,3.656863,5.0,5.015873,5.0
9,1.0,3.0,3.0,1.0,4.0,NaN,NaN,NaN,1.0,308.71697,...,293.24860,282.08571,3.776109,5.0,2.520768,4.0,2.823340,5.0,1.930476,3.0


In [16]:
sample2017 = cycle1_PIAAC2.loc[:, ["GENDER_R", "AGEG10LFS_T", "HOMLANG", "PBROUTE", "PRC_PV_SCR", "PRC_SP_SCR", "PRC_PC_SCR", 
                                   "LITSTATUS", "PVLIT1", "PVLIT2", "PVLIT3", "PVLIT4", "PVLIT5", "PVLIT6", "PVLIT7", "PVLIT8", "PVLIT9", "PVLIT10",
                                   "READHOME", "READHOME_WLE_CA", "READWORK", "READWORK_WLE_CA", 
                                   "WRITHOME", "WRITHOME_WLE_CA", "WRITWORK", "WRITWORK_WLE_CA",]]
sample2017.head(20)

,GENDER_R,AGEG10LFS_T,HOMLANG,PBROUTE,PRC_PV_SCR,PRC_SP_SCR,PRC_PC_SCR,LITSTATUS,PVLIT1,PVLIT2,...,PVLIT9,PVLIT10,READHOME,READHOME_WLE_CA,READWORK,READWORK_WLE_CA,WRITHOME,WRITHOME_WLE_CA,WRITWORK,WRITWORK_WLE_CA
0,1.0,5.0,NaN,4.0,NaN,NaN,NaN,1.0,266.64399,291.69294,...,297.01696,307.50151,2.599661,4.0,NaN,NaN,-0.296028,1.0,NaN,NaN
1,1.0,5.0,NaN,4.0,NaN,NaN,NaN,1.0,305.88773,282.36087,...,314.50435,325.36163,2.838736,5.0,2.943217,5.0,2.675496,4.0,1.523251,2.0
2,2.0,2.0,NaN,4.0,NaN,NaN,NaN,1.0,267.63318,283.43713,...,268.99531,265.87531,2.516906,4.0,NaN,NaN,1.033728,1.0,NaN,NaN
3,2.0,2.0,NaN,4.0,NaN,NaN,NaN,1.0,306.56853,299.24096,...,283.34114,321.50084,2.474552,4.0,2.405418,4.0,2.857830,5.0,1.523251,2.0
4,1.0,3.0,NaN,4.0,NaN,NaN,NaN,1.0,294.78967,320.58617,...,301.74806,301.59756,2.996786,5.0,2.606281,4.0,2.918247,5.0,2.208982,3.0
5,1.0,3.0,NaN,4.0,NaN,NaN,NaN,1.0,207.11040,205.41053,...,172.28108,211.33572,3.022730,5.0,NaN,NaN,NaN,0.0,NaN,NaN
6,2.0,NaN,NaN,3.0,34.0,22.0,44.0,1.0,285.46959,295.44191,...,311.78872,300.41723,1.745798,2.0,NaN,NaN,NaN,0.0,NaN,NaN
7,1.0,4.0,NaN,4.0,NaN,NaN,NaN,1.0,323.49885,318.18011,...,329.14480,348.71943,2.299801,3.0,2.374682,4.0,1.070335,2.0,3.490354,5.0
8,1.0,3.0,NaN,3.0,34.0,22.0,44.0,1.0,316.19245,327.72399,...,333.86587,313.81776,3.158286,5.0,1.606334,2.0,0.420121,1.0,1.003578,1.0
9,2.0,4.0,0.0,1.0,15.0,0.0,0.0,1.0,176.79481,127.97218,...,198.66282,158.98808,-1.178120,1.0,1.992876,3.0,NaN,0.0,NaN,0.0


In [20]:
sample2023 = cycle2_PIAAC.loc[:, ["GENDER_R", "AGEG10LFS", "HOMLANG", "PVLIT1", "PVLIT2", "PVLIT3", "PVLIT4", "PVLIT5", "PVLIT6", "PVLIT7", "PVLIT8", "PVLIT9", "PVLIT10"]]

sample2017.head(20)

,GENDER_R,AGEG10LFS_T,HOMLANG,PBROUTE,PRC_PV_SCR,PRC_SP_SCR,PRC_PC_SCR,LITSTATUS,PVLIT1,PVLIT2,...,PVLIT9,PVLIT10,READHOME,READHOME_WLE_CA,READWORK,READWORK_WLE_CA,WRITHOME,WRITHOME_WLE_CA,WRITWORK,WRITWORK_WLE_CA
0,1.0,5.0,NaN,4.0,NaN,NaN,NaN,1.0,266.64399,291.69294,...,297.01696,307.50151,2.599661,4.0,NaN,NaN,-0.296028,1.0,NaN,NaN
1,1.0,5.0,NaN,4.0,NaN,NaN,NaN,1.0,305.88773,282.36087,...,314.50435,325.36163,2.838736,5.0,2.943217,5.0,2.675496,4.0,1.523251,2.0
2,2.0,2.0,NaN,4.0,NaN,NaN,NaN,1.0,267.63318,283.43713,...,268.99531,265.87531,2.516906,4.0,NaN,NaN,1.033728,1.0,NaN,NaN
3,2.0,2.0,NaN,4.0,NaN,NaN,NaN,1.0,306.56853,299.24096,...,283.34114,321.50084,2.474552,4.0,2.405418,4.0,2.857830,5.0,1.523251,2.0
4,1.0,3.0,NaN,4.0,NaN,NaN,NaN,1.0,294.78967,320.58617,...,301.74806,301.59756,2.996786,5.0,2.606281,4.0,2.918247,5.0,2.208982,3.0
5,1.0,3.0,NaN,4.0,NaN,NaN,NaN,1.0,207.11040,205.41053,...,172.28108,211.33572,3.022730,5.0,NaN,NaN,NaN,0.0,NaN,NaN
6,2.0,NaN,NaN,3.0,34.0,22.0,44.0,1.0,285.46959,295.44191,...,311.78872,300.41723,1.745798,2.0,NaN,NaN,NaN,0.0,NaN,NaN
7,1.0,4.0,NaN,4.0,NaN,NaN,NaN,1.0,323.49885,318.18011,...,329.14480,348.71943,2.299801,3.0,2.374682,4.0,1.070335,2.0,3.490354,5.0
8,1.0,3.0,NaN,3.0,34.0,22.0,44.0,1.0,316.19245,327.72399,...,333.86587,313.81776,3.158286,5.0,1.606334,2.0,0.420121,1.0,1.003578,1.0
9,2.0,4.0,0.0,1.0,15.0,0.0,0.0,1.0,176.79481,127.97218,...,198.66282,158.98808,-1.178120,1.0,1.992876,3.0,NaN,0.0,NaN,0.0


In [19]:


# variableContained1 = list(cycle1_PIAAC1.columns)
# variableWant1 = ["SEQID", "GENDER_R", "DISP_CIBQ", "AGE5GLFSEXT", "AGEG5LFS_T", "AGE5GLFS", "AGEG10LFSXT", "AGEG10LFS_T", "AGEG10LFS", "AGE_R", 
#                  "AGE1634", "AGE6674", "LEAVER1624", "REGION_US", "URBAN_4CAT", "HOMLANG", "FIRLGRGNUS_C", "HOMLGRGNUS_C", "LANGUAGE", ] 

# variableWant2 = ["B_Q01A", "B_Q04A", "B_Q01AUS_C", "D_Q13C", "G_Q04", "G_Q06", "G_Q07", "G_Q08", "J_Q08", "I_Q06DUSX1A", "I_Q06DUSX1B", 
#                  "I_Q06DUSX1C", "I_Q06DUSX1D", "I_Q06DUSX1E", "I_Q06DUSX1F", "I_Q06DUSX1G", "I_Q08USX1", "I_Q08DUSX3", "I_Q10BUSX2A", 
#                  "I_Q10BUSX2B", "I_Q10BUSX2C", "I_Q10BUSX2D", "I_Q10BUSX2E", "I_Q10BUSX2F", "I_Q10BUSX2G", "J_Q05A2USX2", "J_Q05CUSX3D"]

# variableWant3 = ["PBROUTE",   "PRC_PV_SCR", "PRC_SP_SCR", "PRC_PC_SCR", "LITSTATUS", 
#                  "PVLIT1", "PVLIT2", "PVLIT3", "PVLIT4", "PVLIT5", "PVLIT6", "PVLIT7", "PVLIT8", "PVLIT9", "PVLIT10"]

# variableWant4 = ["READYTOLEARN", "READYTOLEARN_WLE_CA", "ICTHOME", "ICTHOME_WLE_CA", "ICTWORK", "ICTWORK_WLE_CA", "READHOME", "READHOME_WLE_CA", 
#                  "READWORK", "READWORK_WLE_CA", "WRITHOME", "WRITHOME_WLE_CA", "WRITWORK", "WRITWORK_WLE_CA"]

# variableWantExtra = ["G_Q01A", "G_Q01B", "G_Q01C", "G_Q01D", "G_Q01E", "G_Q01F", "G_Q02A", "G_Q02B", "G_Q02C", "G_Q02D",
#                      "G_Q05A", "G_Q05C", "G_Q05H", "H_Q01A", "H_Q01B", "H_Q01C", "H_Q01D", "H_Q01E", "H_Q01F", 
#                      "H_Q02A", "H_Q02B", "H_Q02C", "H_Q02D", "H_Q05A", "H_Q05C", "H_Q05H", "I_Q04D", "I_Q04M"]

# newList1 = []

# for x in variableWant1:
#    if x in variableContained1:
#        print(f"{x}: available")
#        newList1.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant2:
#    if x in variableContained1:
#        print(f"{x}: available")
#        newList1.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant3:
#    if x in variableContained1:
#        print(f"{x}: available")
#        newList1.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant4:
#    if x in variableContained1:
#        print(f"{x}: available")
#        newList1.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWantExtra:
#    if x in variableContained1:
#        print(f"{x}: available")
#        newList1.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

In [18]:

# newList2 = []
# variableContained2 = list(cycle1_PIAAC2.columns)

# for x in variableWant1:
#    if x in variableContained2:
#        print(f"{x}: available")
#        newList2.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant2:
#    if x in variableContained2:
#        print(f"{x}: available")
#        newList2.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant3:
#    if x in variableContained2:
#        print(f"{x}: available")
#        newList2.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant4:
#    if x in variableContained2:
#        print(f"{x}: available")
#        newList2.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWantExtra:
#    if x in variableContained2:
#        print(f"{x}: available")
#        newList2.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

In [17]:

# newList3 = []
# variableContained3 = list(cycle2_PIAAC.columns)

# for x in variableWant1:
#    if x in variableContained3:
#        print(f"{x}: available")
#        newList3.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant2:
#    if x in variableContained3:
#        print(f"{x}: available")
#        newList3.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant3:
#    if x in variableContained3:
#        print(f"{x}: available")
#        newList3.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWant4:
#    if x in variableContained3:
#        print(f"{x}: available")
#        newList3.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")

# for x in variableWantExtra:
#    if x in variableContained3:
#        print(f"{x}: available")
#        newList3.append(x)
#    # else:
#    #     print(f"{x}: [UN-available]")